# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: fetch latest code</h2>
            <span style="color:#f71;">I'm continually improving these labs, adding more examples and exercises.
            At the start of each week, it's worth checking you have the latest code.<br/>
            First do a <a href="https://chatgpt.com/share/6734e705-3270-8012-a074-421661af6ba9">git pull and merge your changes as needed</a>. Any problems? Try asking ChatGPT to clarify how to merge - or contact me!<br/><br/>
            After you've pulled the code, from the llm_engineering directory, in a Cursor Terminal, run:<br/>
            <code>uv sync</code><br/>
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            In this lab, I use high end models GPT 5, Claude 4.5 Sonnet, Gemini 2.5 Pro, Grok 4, which are the slightly higher priced models. The costs are still low, but if you'd prefer to keep costs ultra low, please pick lower cost models like gpt-5-nano.
            </span>
        </td>
    </tr>
</table>

In [1]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
deepseek_api_key=os.getenv('DEEPSEEK_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if deepseek_api_key:
    print(f"Deepseek API Key exists and begins {deepseek_api_key[:7]}")
else:
    print("Deepseek API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Deepseek API Key exists and begins sk-079b
Google API Key exists and begins AQ
Groq API Key exists and begins gsk_


In [3]:
# Connect to client libraries

openai = OpenAI()

deepseek_url = "https://api.deepseek.com/v1"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
groq_url = "https://api.groq.com/openai/v1"

deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)

In [4]:
OPENAI_MODEL = "gpt-5"
DEEPSEEK_MODEL = "deepseek-chat"   # or "deepseek-reasoner"
GROQ_MODEL = "llama-3.3-70b-versatile"   # pick any Groq-hosted model
GEMINI_MODEL = "gemini-2.5-flash"

# Want to keep costs ultra-low? Uncomment these lines:

# OPENAI_MODEL = "gpt-5-nano"
# CLAUDE_MODEL = "claude-haiku-4-5"
# GROK_MODEL = "grok-4-fast-non-reasoning"
# GEMINI_MODEL = "gemini-2.5-flash-lite"

## PLEASE NOTE:

We will be writing a solution to convert Python into efficient, optimized C++ code for your machine, which can be compiled to native machine code and executed.

It is not necessary for you to execute the code yourself - that's not the point of the exercise!

But if you would like to (because it's satisfying!) then I'm including the steps here. Very optional!

As an alternative, I'll also show you a website where you can run the C++ code.

In [5]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '10',
  'version': '10.0.19045',
  'kernel': '10',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': ''},
 'package_managers': ['winget'],
 'cpu': {'brand': 'Intel(R) Core(TM) i5-7200U CPU @ 2.50GHz',
  'cores_logical': 4,
  'cores_physical': 2,
  'simd': []},
 'toolchain': {'compilers': {'gcc': '', 'g++': '', 'clang': '', 'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

In [6]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = openai.chat.completions.create(model=OPENAI_MODEL, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))
    

Short answer: you don’t have any C++ compiler installed yet.

Simplest way to get one on your Windows 10 system (using winget):
1) Install MSYS2
- Open PowerShell (Administrator) and run:
  winget install -e --id MSYS2.MSYS2

2) Install a native Windows GCC (MinGW-w64, UCRT64) inside MSYS2
- Run:
  "C:\msys64\usr\bin\bash.exe" -lc "pacman -Sy --noconfirm && pacman -S --noconfirm --needed mingw-w64-ucrt-x86_64-gcc"
- Verify:
  "C:\msys64\ucrt64\bin\g++" --version

That’s it—you can now compile C++.

Python commands (fastest runtime focus)
- Put main.cpp in your current working directory for the Python script.
- Use these commands (absolute path avoids PATH setup):

compile_command = [
    r"C:\msys64\ucrt64\bin\g++.exe",
    "-std=c++20",
    "-O3",
    "-march=native",
    "-flto",
    "-DNDEBUG",
    "main.cpp",
    "-o", "main.exe"
]
run_command = ["main.exe"]

Notes:
- -O3 -march=native -flto aims for maximum runtime performance on your CPU. It may increase compile time and produce an executable optimized for this machine (not necessarily portable to much older CPUs).
- If you want to avoid any dependency on MSYS2 DLLs, you can add "-static-libstdc++" and "-static-libgcc" to the compile command.

## If you need to install something

If you would like to, please follow GPTs instructions! Then rerun the analysis afterwards (you might need to Restart the notebook) to confirm you're set.

You should now be equipped with the command to compile the code, and the command to run it!

Enter that in the cell below:

In [52]:
import subprocess

# This runs the installation directly inside your MSYS2 folder from Python
install_command = [
    r"C:\msys64\usr\bin\bash.exe", 
    "-lc", 
    "pacman -Sy --noconfirm && pacman -S --noconfirm --needed mingw-w64-ucrt-x86_64-gcc"
]

print("Installing compiler... Please wait (this can take up to a minute)...")
result = subprocess.run(install_command, capture_output=True, text=True)

if result.returncode == 0:
    print("🎉 SUCCESS! The C++ compiler is now installed inside MSYS2.")
else:
    print("❌ Something went wrong:")
    print(result.stderr)

Installing compiler... Please wait (this can take up to a minute)...


FileNotFoundError: [WinError 2] Sistem belirtilen dosyayı bulamıyor

In [49]:
# Direct paths bypass the cmd.exe string-parsing bugs entirely
compile_command = [
    r"C:\Program Files (x86)\Microsoft Visual Studio\2022\BuildTools\VC\Tools\MSVC\14.40.33807\bin\Hostx64\x64\cl.exe", 
    "/nologo", "/O2", "/GL", "/arch:AVX2", "/EHsc", "/std:c++20", "/Fe:main.exe", "main.cpp", "/link", "/LTCG"
]

run_command = ["main.exe"]

## And now, on with the main task

In [8]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [9]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [10]:
def write_output(cpp):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp)

In [11]:
def port(client, model, python):
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)

In [12]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [13]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [14]:
run_python(pi)

Result: 3.141592656089
Execution Time: 130.416046 seconds


In [15]:
port(openai, OPENAI_MODEL, pi)

# Compiling C++ and executing

This next cell contains the command to compile a C++ file based on the instructions from GPT.

Again, it's not crucial to do this step if you don't wish to!

OR alternatively: student Sandeep K.G. points out that you can run Python and C++ code online to test it out that way. Thank you Sandeep!  
> Not an exact comparison but you can still get the idea of performance difference.  
> For example here: https://www.programiz.com/cpp-programming/online-compiler/

In [50]:
import subprocess
import os
import glob

def compile_and_run():
    global compile_command
    
    # Dynamic path fixer in case your exact MSVC version number is different
    base_path = r"C:\Program Files (x86)\Microsoft Visual Studio\2022\BuildTools\VC\Tools\MSVC"
    if os.path.exists(base_path):
        versions = glob.glob(os.path.join(base_path, "*"))
        if versions:
            latest_version = max(versions, key=os.path.getmtime)
            cl_path = os.path.join(latest_version, "bin", "Hostx64", "x64", "cl.exe")
            if os.path.exists(cl_path):
                compile_command[0] = cl_path

    try:
        # shell=False passes the direct executable cleanly
        subprocess.run(compile_command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, shell=False)
        print("🎉 Compilation successful!")
    except subprocess.CalledProcessError as e:
        print("❌ --- COMPILER ERROR OUTPUT ---")
        stdout_decoded = e.stdout.decode('cp437', errors='replace') if e.stdout else ""
        stderr_decoded = e.stderr.decode('cp437', errors='replace') if e.stderr else ""
        print("STDOUT:\n", stdout_decoded)
        print("STDERR:\n", stderr_decoded)
        return

    # Run 3 separate times
    for i in range(3):
        result = subprocess.run(run_command, check=True, text=True, capture_output=True, shell=False)
        print(f"--- Run {i+1} ---")
        print(result.stdout)

In [51]:
compile_and_run()

❌ --- COMPILER ERROR OUTPUT ---
STDOUT:
 main.cpp
main.cpp(2): fatal error C1034: iostream: hiçbir ekleme yolu ayarlanmadì

STDERR:
 


In [38]:
19.178207/0.082168

233.40238292279233

## OK let's try the other contenders!

In [39]:
print(DEEPSEEK_MODEL)
print(GROQ_MODEL)
print(GEMINI_MODEL)

deepseek-chat
llama-3.3-70b-versatile
gemini-2.5-flash


In [ ]:
from google import genai

client = genai.Client(
    api_key=os.getenv("GOOGLE_API_KEY")
)

In [40]:
port(deepseek, DEEPSEEK_MODEL, pi)
compile_and_run()

APIStatusError: Error code: 402 - {'error': {'message': 'Insufficient Balance', 'type': 'unknown_error', 'param': None, 'code': 'invalid_request_error'}}

In [41]:
port(groq, GROQ_MODEL, pi)
compile_and_run()

❌ --- COMPILER ERROR OUTPUT ---
STDOUT:
 
STDERR:
 '\"C:\Program Files (x86)\Microsoft Visual Studio\2022\BuildTools\VC\Auxiliary\Build\vcvars64.bat\"' is not recognized as an internal or external command,
operable program or batch file.



In [42]:
port(gemini, GEMINI_MODEL, pi)
compile_and_run()


❌ --- COMPILER ERROR OUTPUT ---
STDOUT:
 
STDERR:
 '\"C:\Program Files (x86)\Microsoft Visual Studio\2022\BuildTools\VC\Auxiliary\Build\vcvars64.bat\"' is not recognized as an internal or external command,
operable program or batch file.



In [ ]:
print(f"""
In Ed's experiments, the performance speedups were:

4th place: DeepSeek 4.5: {19.178207/0.104241:.0f}X speedup
3rd place: GPT-5: {19.178207/0.082168:.0f}X speedup
2nd place: Groq 4: {19.178207/0.018092:.0f}X speedup
1st place: Gemini 2.5 Pro: {19.178207/0.013314:.0f}X speedup
""")